In [210]:
import pandas as pd
df = pd.read_csv('dataset.csv')
print("Размер данных:", df.shape)
print("\nПервые 3 строки:")
df['budget'] = df['budget']/20000 ##попытка нормализации
df['taxi'] = df['taxi']/5 ##попытка нормализации
df.head(3)

Размер данных: (33, 21)

Первые 3 строки:


,os,sex,siblings,osPC,taxi,mobgames,regionMos,camera,regionRF,pay,...,occupation,smarthouse,IT,watches,budget,charges,browser,technologies,interface,material
0,1,1,1,1,4.0,1,5,1,1,1,...,2,2,2,2,6.0,2,2,4,2,1
1,1,2,0,2,3.0,2,4,1,6,1,...,2,2,1,3,6.0,1,3,3,2,1
2,1,1,2,1,0.2,2,4,1,1,1,...,3,1,1,3,5.0,2,2,5,1,1


In [ ]:
x = df.drop('os', axis=1) ##атрибуты
y = df['os'] ##целевой столбец
border = df.shape[0]*0.7 # ограничение обучающей выборки - 70%
dft = df.loc[:border, :]
x_train = x.loc[:border, :]
y_train = y.loc[:border]
x_test = x.loc[border:,:]
y_test = y.loc[border:] ##разделение на тестовую и обучающую выборки
print(y_test)

24    2
25    2
26    1
27    2
28    1
29    2
30    2
31    2
32    1
Name: os, dtype: int64


In [212]:
def calculate_distances(row):
    return (abs(x_test - row)).sum(axis=1)

distances = x_train.apply(calculate_distances, axis=1)

distances.insert(0, 'OS', y_train)
print(distances) ##каждый столбец - test, строка - train, в каждой ячейке - манхэттенское расстояние

    OS     24     25     26     27     28     29     30     31     32
0    1  19.00  25.00  16.80  22.00  21.00  21.50  22.50  15.10  13.40
1    1  19.00  21.00  20.80  30.00  33.00  23.50  24.50  19.10  19.40
2    1  15.20  25.20  17.00  16.20  24.80  15.70  22.30  15.70  12.40
3    2  18.50  18.50  16.70  15.50  32.50   8.00  21.00  18.40  16.10
4    1  10.50  21.50  12.70  18.50  26.50  12.00  17.00  12.40  10.10
5    2  10.40  20.40  14.20  17.40  27.60  10.90  19.10  12.50  11.20
6    1  14.60  20.60  18.40  15.60  25.40  17.10  14.90  12.70  13.00
7    1  15.10  23.10  11.90  22.10  29.90  19.60  19.40  15.20  11.50
8    2  17.00  11.00  17.20  14.00  27.00  11.50  15.50   8.90  14.60
9    1  20.10  12.10  17.90   9.10  30.90  13.60  13.40  11.20  14.50
10   1  17.00  19.00  23.20  18.00  33.00  23.50  24.50  15.90  16.60
11   2  23.15  17.15  22.95  16.15  28.35  14.15  18.85  16.25  19.95
12   1  11.60  21.60  15.40  16.60  26.40  18.10  17.90   9.70  12.00
13   2  20.45  14.45

In [213]:
def calculate_accuracy(df1, df2):   
    # Сравниваем все значения
    df1_reset = df1.reset_index(drop=True)
    df2_reset = df2.reset_index(drop=True)
    matches = (df1_reset == df2_reset).sum() ## если предсказание совпадает с test, то true, потом складываем количество true
    total = df1_reset.size #общее количество строк
    accuracy = (matches / total) * 100 # точность в процентах
    
    return accuracy

In [214]:
j = 0
eval = pd.DataFrame(columns=['k','acc']) #датафрейм с результатами для разных k
for k in range (1, 10, 2):    ##перебор нечетных k с 1 по 9
    results = []
    for i in range(1, distances.shape[1]):
        data = distances.iloc[:, [0, i]] ##берем целевой столбец OS и i-тый стобец с расстояниями
        data = data.sort_values(by=data.columns[1]) #сортируем по возрастанию, чтобы верхние были ближайшими
        k_n = data.head(k) ##берем k ближайших
        OS = k_n.iloc[:, 0].sum(axis=0) / k_n.shape[0] # складываем все строки столбца OS и находим среднее значение предсказания
        OS = round(OS) #округляем предсказание до целого значения
        results.append(OS) # формируем массив
    answer = pd.Series(results)
    acc = calculate_accuracy(answer, y_test)
    
    eval.loc[j] = [k, acc]
    j = j+1
print(eval)

     k        acc
0  1.0  88.888889
1  3.0  77.777778
2  5.0  77.777778
3  7.0  77.777778
4  9.0  77.777778


# В итоге модель как будто не предсказывает, а просто длинным путем показывает нам статистику - ~70-80% людей ходят с андроидом, можно было бы просто всем давать предсказание "Андроид" и попадать в большинстве случаев